In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
# Đọc file CSV trực tiếp vào biến df
df = pd.read_csv("web_traffic.csv" , sep=";")

print("5 DÒNG ĐẦU TIÊN FILE WEB TRAFFIC")
display(df.head())

print("\nTHÔNG TIN CÁC CỘT FILE WEB TRAFFIC")
df.info()

5 DÒNG ĐẦU TIÊN FILE WEB TRAFFIC


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,01/01/2013,9760,7253,39093,0.00514,102.9,organic_search
1,02/01/2013,10456,8151,47611,0.00406,120.5,organic_search
2,03/01/2013,10076,7458,36963,0.00401,263.6,direct
3,04/01/2013,9973,8063,53078,0.00562,151.8,direct
4,05/01/2013,10223,7882,36790,0.00525,168.6,referral



THÔNG TIN CÁC CỘT FILE WEB TRAFFIC
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      3652 non-null   object 
 1   sessions                  3652 non-null   int64  
 2   unique_visitors           3652 non-null   int64  
 3   page_views                3652 non-null   int64  
 4   bounce_rate               3652 non-null   float64
 5   avg_session_duration_sec  3652 non-null   float64
 6   traffic_source            3652 non-null   object 
dtypes: float64(2), int64(3), object(2)
memory usage: 199.8+ KB


In [3]:
print("Số lượng giá trị NULL từng cột:")
print(df.isna().sum())

print("\nNgày có duy nhất không:", df["date"].is_unique)
display(df[df["date"].duplicated(keep=False)])

Số lượng giá trị NULL từng cột:
date                        0
sessions                    0
unique_visitors             0
page_views                  0
bounce_rate                 0
avg_session_duration_sec    0
traffic_source              0
dtype: int64

Ngày có duy nhất không: True


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source


In [4]:
# Chuẩn hóa ngày và các cột số
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")
number_cols = ["sessions", "unique_visitors", "page_views", "bounce_rate", "avg_session_duration_sec"]

for col in number_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[number_cols] = df[number_cols].replace([np.inf, -np.inf], np.nan)

# Làm tròn để tránh sai số khi so sánh dữ liệu
df["bounce_rate"] = df["bounce_rate"].round(8)
df["avg_session_duration_sec"] = df["avg_session_duration_sec"].round(6)

df["traffic_source"] = df["traffic_source"].astype("string").str.strip().str.lower()
df["traffic_source"] = df["traffic_source"].replace("", pd.NA)

# Kiểm tra danh sách nguồn truy cập hợp lệ
valid_sources = ["direct", "email_campaign", "organic_search", "paid_search", "referral", "social_media"]

print("Phân bố nguồn truy cập:")
display(df["traffic_source"].value_counts(dropna=False))
display(df[~df["traffic_source"].isin(valid_sources)])

Phân bố nguồn truy cập:


traffic_source
organic_search    1090
paid_search        784
social_media       632
email_campaign     505
referral           375
direct             266
Name: count, dtype: Int64

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source


In [5]:
# Đọc file tf.json trực tiếp từ thư mục hiện tại
with open("tf.json", "r", encoding="utf-8") as f:
    tf = pd.DataFrame(json.load(f))

print("5 DÒNG ĐẦU TIÊN FILE TF JSON")
display(tf.head())

print("\nTHÔNG TIN CÁC CỘT FILE TF JSON")
tf.info()

5 DÒNG ĐẦU TIÊN FILE TF JSON


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-10-26,12422,8842,45914,0.00452,197.1,organic_search Variant 0001
1,2019-02-15,26260,19148,97929,0.00568,116.5,paid_search Variant 0002
2,2019-01-12,14383,11480,57737,0.00466,170.6,email_campaign Variant 0003
3,2017-09-30,22021,16969,95903,0.00344,111.8,paid_search Variant 0004
4,2021-08-12,38269,29771,208637,0.00442,210.7,paid_search Variant 0005



THÔNG TIN CÁC CỘT FILE TF JSON
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      1000 non-null   object 
 1   sessions                  999 non-null    object 
 2   unique_visitors           1000 non-null   object 
 3   page_views                1000 non-null   int64  
 4   bounce_rate               1000 non-null   float64
 5   avg_session_duration_sec  1000 non-null   float64
 6   traffic_source            1000 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 54.8+ KB


In [6]:
tf_raw = tf.copy()
tf["date"] = pd.to_datetime(tf["date"], format="%Y-%m-%d", errors="coerce")

for col in number_cols:
    tf[col] = pd.to_numeric(tf[col], errors="coerce")

tf[number_cols] = tf[number_cols].replace([np.inf, -np.inf], np.nan)
tf["bounce_rate"] = tf["bounce_rate"].round(8)
tf["avg_session_duration_sec"] = tf["avg_session_duration_sec"].round(6)

print("Số lượng NULL trong file TF sau khi ép kiểu:")
print(tf.isna().sum())

# Làm sạch chuỗi traffic_source và xử lý hậu tố "Variant"
tf["traffic_source"] = tf["traffic_source"].astype("string").str.strip().str.lower()
tf["traffic_source"] = tf["traffic_source"].str.replace(r"\s+", " ", regex=True)

source = tf["traffic_source"].str.replace(r"\s+variant\s+\d+$", "", regex=True)
mask = source.isin(valid_sources)
tf.loc[mask, "traffic_source"] = source[mask]
tf["traffic_source"] = tf["traffic_source"].replace(["", "n/a", "null", "none"], pd.NA)

print("\nPhân bố nguồn truy cập file TF:")
display(tf["traffic_source"].value_counts(dropna=False))

Số lượng NULL trong file TF sau khi ép kiểu:
date                        2
sessions                    2
unique_visitors             1
page_views                  0
bounce_rate                 0
avg_session_duration_sec    0
traffic_source              0
dtype: int64

Phân bố nguồn truy cập file TF:


traffic_source
organic_search    302
paid_search       220
social_media      180
email_campaign    132
referral          102
direct             62
<NA>                2
Name: count, dtype: Int64

In [7]:
# 1. Ngày bị thiếu/lỗi (NaT)
missing_date = tf[tf["date"].isna()].reset_index()
cols_date = ["sessions", "unique_visitors", "page_views", "bounce_rate", "avg_session_duration_sec", "traffic_source"]
matched_date = missing_date.merge(df[cols_date].drop_duplicates(), on=cols_date, how="inner")
tf = tf.drop(index=matched_date["index"])

# 2. Sessions bị thiếu (NaN)
missing_sess = tf[tf["sessions"].isna()].reset_index()
cols_sess = ["date", "unique_visitors", "page_views", "bounce_rate", "avg_session_duration_sec", "traffic_source"]
matched_sess = missing_sess.merge(df[cols_sess].drop_duplicates(), on=cols_sess, how="inner")
tf = tf.drop(index=matched_sess["index"])

# 3. Sessions bị âm (< 0)
missing_neg = tf[tf["sessions"] < 0].reset_index()
cols_neg = ["date", "unique_visitors", "page_views", "bounce_rate", "avg_session_duration_sec", "traffic_source"]
matched_neg = missing_neg.merge(df[cols_neg].drop_duplicates(), on=cols_neg, how="inner")
tf = tf.drop(index=matched_neg["index"])

# 4. Sessions lớn hơn Page Views
missing_pv = tf[tf["sessions"] > tf["page_views"]].reset_index()
cols_pv = ["date", "unique_visitors", "page_views", "bounce_rate", "avg_session_duration_sec", "traffic_source"]
matched_pv = missing_pv.merge(df[cols_pv].drop_duplicates(), on=cols_pv, how="inner")
tf = tf.drop(index=matched_pv["index"])

# 5. Unique Visitors bị thiếu (NaN)
missing_uv = tf[tf["unique_visitors"].isna()].reset_index()
cols_uv = ["date", "sessions", "page_views", "bounce_rate", "avg_session_duration_sec", "traffic_source"]
matched_uv = missing_uv.merge(df[cols_uv].drop_duplicates(), on=cols_uv, how="inner")
tf = tf.drop(index=matched_uv["index"])

# 6. Unique Visitors bị âm (< 0)
missing_uv_neg = tf[tf["unique_visitors"] < 0].reset_index()
cols_uv_neg = ["date", "sessions", "page_views", "bounce_rate", "avg_session_duration_sec", "traffic_source"]
matched_uv_neg = missing_uv_neg.merge(df[cols_uv_neg].drop_duplicates(), on=cols_uv_neg, how="inner")
tf = tf.drop(index=matched_uv_neg["index"])

# 7. Nguồn truy cập bị thiếu/không hợp lệ
missing_src = tf[~tf["traffic_source"].isin(valid_sources)].reset_index()
cols_src = ["date", "sessions", "unique_visitors", "page_views", "bounce_rate", "avg_session_duration_sec"]
matched_src = missing_src.merge(df[cols_src].drop_duplicates(), on=cols_src, how="inner")
tf = tf.drop(index=matched_src["index"])

# Bỏ trùng hoàn toàn & reset index
tf = tf.drop_duplicates()
duplicate_dates = tf[tf["date"].duplicated(keep=False)].copy()
tf = tf.drop(index=duplicate_dates.index).reset_index(drop=True)

print("Số dòng TF còn lại để so sánh:", len(tf))

Số dòng TF còn lại để so sánh: 998


In [8]:
# So sánh tất cả các cột giữa TF và df
compare_cols = ["date", "sessions", "unique_visitors", "page_views", "bounce_rate", "avg_session_duration_sec", "traffic_source"]
merged = tf.merge(df[compare_cols].drop_duplicates(), on=compare_cols, how="left", indicator=True)

already_exists = merged[merged["_merge"] == "both"]
not_matched = merged[merged["_merge"] == "left_only"].drop(columns="_merge")

print("TF đã có trong Web:", len(already_exists))
print("TF chưa khớp:", len(not_matched))

# Lấy các ngày mới chưa tồn tại trong df (nếu có)
new_from_tf = not_matched[~not_matched["date"].isin(df["date"])].copy()
print("Ngày mới được thêm:", len(new_from_tf))

# Gộp dữ liệu mới vào df
df = pd.concat([df, new_from_tf], ignore_index=True)
df = df.sort_values("date").reset_index(drop=True)

for col in ["sessions", "unique_visitors", "page_views"]:
    df[col] = df[col].astype("Int64")

print("\n--- THÔNG TIN KẾT QUẢ CUỐI CÙNG (DF) ---")
print("Tổng số dòng:", len(df))
print("Số ngày trùng:", df["date"].duplicated().sum())
print("Số null:", df.isna().sum().sum())
df.info()

# Xuất file CSV từ biến df
output_path = "web_traffic_cleaned.csv"
df.to_csv(output_path, index=False,sep=',', encoding="utf-8-sig")
print(f"\nĐã xuất file thành công tại: {output_path}")

TF đã có trong Web: 0
TF chưa khớp: 998
Ngày mới được thêm: 998

--- THÔNG TIN KẾT QUẢ CUỐI CÙNG (DF) ---
Tổng số dòng: 4650
Số ngày trùng: 3651
Số null: 3657
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4650 entries, 0 to 4649
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      998 non-null    datetime64[ns]
 1   sessions                  4648 non-null   Int64         
 2   unique_visitors           4649 non-null   Int64         
 3   page_views                4650 non-null   Int64         
 4   bounce_rate               4650 non-null   float64       
 5   avg_session_duration_sec  4650 non-null   float64       
 6   traffic_source            4648 non-null   string        
dtypes: Int64(3), datetime64[ns](1), float64(2), string(1)
memory usage: 268.0 KB

Đã xuất file thành công tại: web_traffic_cleaned.csv
